# ConvTran Reproduction: BasicMotions Pilot

We reproduce ConvTran from *Improving Position Encoding of Transformers for Multivariate Time Series Classification* by Foumani et al.

We start with BasicMotions to check the training pipeline before running the remaining UEA datasets. This notebook records the setup, configurationand results of that first experiment.

- Paper: https://doi.org/10.1007/s10618-023-00948-2
- Official code: https://github.com/Navidfoumani/ConvTran
- Our repository: https://github.com/tyrerodr/convtran-reproduction

We use `scripts/run_baseline.py` to run the experiments and collect their results in `results/baseline_runs.csv`.

In [23]:
from pathlib import Path
import platform
import subprocess
import sys

import pandas as pd
import torch

ROOT = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "upstream" / "ConvTran" / "main.py").is_file()
    ),
    None,
)

assert ROOT is not None, "Open the notebook from the project repository."

UPSTREAM = ROOT / "upstream" / "ConvTran"

print("Repository:", ROOT)
print("Kernel Python:", sys.executable)

Repository: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction
Kernel Python: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/.venv/bin/python


## Environment

The table below shows the environment used to run this notebook and the commit of the official ConvTran implementation.

Training configurations, logs and checkpoints are saved in `outputs/`.

In [24]:
upstream_commit = subprocess.check_output(
    ["git", "-C", str(UPSTREAM), "rev-parse", "HEAD"],
    text=True,
).strip()

environment = {
    "Python": platform.python_version(),
    "PyTorch": torch.__version__,
    "PyTorch CUDA build": torch.version.cuda,
    "CUDA available": torch.cuda.is_available(),
    "GPU": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "Not available"
    ),
    "Upstream commit": upstream_commit,
}

pd.DataFrame(environment.items(), columns=["Component", "Value"])

,Component,Value
0,Python,3.10.19
1,PyTorch,2.0.1+cu118
2,PyTorch CUDA build,11.8
3,CUDA available,True
4,GPU,NVIDIA GeForce RTX 4070 Ti
5,Upstream commit,148afb6ca549915b7e78b05e2ec3b4ba6e052341


## Baseline protocol

For this first experiment, we use the released ConvTran implementation without changing its architecture. The settings are listed below.

| Setting | Value |
|---|---|
| Model | ConvTran |
| Absolute positional encoding | tAPE |
| Relative positional encoding | eRPE |
| Embedding dimension | 16 |
| Attention heads | 8 |
| Feedforward dimension | 256 |
| Optimizer | RAdam |
| Learning rate | 0.001 |
| Dropout | 0.01 |
| Epochs | 100 |
| Batch size | 16 |
| Normalization option | Disabled |
| Validation ratio | 0.2 |
| Training seeds | 1234, 2345, 3456, 4567, 5678 |
| Checkpoint selection | Lowest validation loss |

We run five training seeds, each in a separate process, and evaluate the checkpoint with the lowest validation loss.

The official loader uses a fixed training/validation split (`random_state=1234`) and caches the processed data. The five runs therefore share the same split.

These settings follow the code defaults. Some differ from the paper, as discussed later in this notebook.

## BasicMotions

BasicMotions contains six channels and four classes. We use the official UEA train/test split and reserve 20% of the training set for validation. This gives us 32 training, 8 validation and 40 test samples.

We chose this dataset for the initial check because it is small and quick to train.

## Running the experiment

The following cell runs the baseline script for BasicMotions.

The script reuses completed runs when their logs are available.
Otherwise, it starts training and saves the output for each seed.

Our first five runs were completed from the terminal. The script
collected those results without repeating the training.

In [25]:
command = [
    sys.executable,
    str(ROOT / "scripts" / "run_baseline.py"),
    "--dataset",
    "BasicMotions",
]

print("Command:", " ".join(command))

subprocess.run(
    command,
    cwd=ROOT,
    check=True,
)

Command: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/.venv/bin/python /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/scripts/run_baseline.py --dataset BasicMotions
Existing result: BasicMotions, seed=1234
Existing result: BasicMotions, seed=2345
Existing result: BasicMotions, seed=3456
Existing result: BasicMotions, seed=4567
Existing result: BasicMotions, seed=5678
Saved results to: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/results/baseline_runs.csv


CompletedProcess(args=['/home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/.venv/bin/python', '/home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/scripts/run_baseline.py', '--dataset', 'BasicMotions'], returncode=0)

In [26]:
results_path = ROOT / "results" / "baseline_runs.csv"

assert results_path.is_file(), (
    "Results file not found. Run scripts/run_baseline.py first."
)

runs = pd.read_csv(results_path)

required_columns = {
    "dataset",
    "seed",
    "epochs",
    "batch_size",
    "accuracy",
    "test_loss",
}

assert required_columns.issubset(runs.columns), (
    "The results CSV is missing required columns."
)
assert not runs.duplicated(["dataset", "seed"]).any(), (
    "Duplicate dataset/seed entries found."
)
assert runs["accuracy"].between(0, 1).all(), (
    "Accuracy values must be between 0 and 1."
)

runs = runs.sort_values(["dataset", "seed"]).reset_index(drop=True)
runs

,dataset,seed,epochs,batch_size,accuracy,test_loss
0,BasicMotions,1234,100,16,1.0,0.161723
1,BasicMotions,2345,100,16,1.0,0.183982
2,BasicMotions,3456,100,16,1.0,0.210695
3,BasicMotions,4567,100,16,1.0,0.159011
4,BasicMotions,5678,100,16,1.0,0.182155


In [27]:
summary = (
    runs.groupby("dataset")
    .agg(
        runs=("seed", "count"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
    )
    .reset_index()
)

summary["accuracy_mean_pct"] = summary["accuracy_mean"] * 100
summary["accuracy_std_pp"] = summary["accuracy_std"] * 100

summary[
    ["dataset", "runs", "accuracy_mean_pct", "accuracy_std_pp"]
].round(4)

,dataset,runs,accuracy_mean_pct,accuracy_std_pp
0,BasicMotions,5,100.0,0.0


## Results

- All five runs correctly classified the 40 test samples. Mean accuracy was 100%, with a standard deviation of 0 percentage points.

- Test loss differed between seeds, even though the predicted classes were the same. This is possible because the models assigned different probabilities to those predictions.

- This result applies to BasicMotions with a fixed data split. We still need to evaluate the remaining datasets.

## Differences between the paper and code

While checking the implementation, we found the differences listed below.
We kept the released code unchanged for this baseline.

| Aspect | Paper description | Released implementation |
|---|---|---|
| Embedding dimension | 64 | Default: 16 |
| Optimizer | Adam | RAdam |
| Training termination | Early stopping based on validation loss | Fixed epoch budget; best validation-loss checkpoint retained |
| Feedforward activation | GELU | ReLU |

Setting the embedding dimension to 64 would also change the number of temporal filters, since the code defines them as `emb_size * 4`. We therefore cannot match the paper's architecture by changing thatargument alone.

In issue #6, the author recommends running datasets separately and averaging five experiments. The response does not fully clarify the configuration differences.

https://github.com/Navidfoumani/ConvTran/issues/6

The author also discusses memory problems with EigenWorms in issue #11. We will check its requirements before attempting the full benchmark.

https://github.com/Navidfoumani/ConvTran/issues/11

## Next steps

We will download the remaining UEA datasets, check their dimensions and run them individually using the same baseline settings. Also, we will compare our results with the paper before evaluating our proposed improvement.

## Limitations

So far, we have evaluated only BasicMotions. The five seeds share one training/validation split, and we have not checked whether the results remain identical across different hardware or software versions.

The baseline uses the released-code defaults, which differ in some respects from the paper's description.